# 🚀 DeepSeek-V3.2 Model with the Foundry OpenAI Client 🧠

**DeepSeek-V3.2** is a state-of-the-art reasoning model combining reinforcement learning and supervised fine-tuning, excelling at complex reasoning tasks with 37B active parameters and 128K context window.

In this notebook, you'll learn to:
1. **Initialize** the Foundry project's OpenAI-compatible client (`get_openai_client()`)
2. **Chat** with DeepSeek-V3.2 using reasoning extraction
3. **Implement** a travel planning example with step-by-step reasoning
4. **Leverage** the 128K context window for complex scenarios

## Why DeepSeek-V3.2?
- **Advanced Reasoning**: Specializes in chain-of-thought problem solving
- **Massive Context**: 128K token window for detailed analysis
- **Efficient Architecture**: 37B active parameters from 671B total
- **Safety Integrated**: Built-in content filtering capabilities


## 1. Setup & Authentication

Required packages:
- `azure-ai-projects` (endpoint-based client) + `openai` (via `get_openai_client()`)
- `azure-identity`: For `DefaultAzureCredential`
- `python-dotenv`: For environment variables

.env file requirements:
```bash
PROJECT_ENDPOINT=<your-project-endpoint>
MODEL_NAME=DeepSeek-V3.2
```

In [ ]:
import os
import re
from dotenv import load_dotenv
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# DeepSeek deployment name (from Foundry > Models + endpoints)
model_name = os.getenv("MODEL_NAME", "DeepSeek-V3.2")

# Initialize client
try:
    # Endpoint-based client + its OpenAI client. In the New Foundry, DeepSeek is a
    # model deployment reached through the same OpenAI-compatible endpoint by name —
    # no separate serverless inference endpoint/key needed.
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Client initialized | Model:", model_name)
except Exception as e:
    print("❌ Initialization failed:", e)

## 2. Intelligent Travel Planning ✈️

Demonstrate DeepSeek-V3.2's reasoning capabilities for trip planning:

In [ ]:
def plan_trip_with_reasoning(query, show_thinking=False):
    """Get travel recommendations with reasoning extraction"""
    messages = [
        {"role": "system", "content": "You are a travel expert. Provide detailed plans with rationale."},
        {"role": "user", "content": f"{query} Include hidden gems and safety considerations."},
    ]

    response = openai_client.chat.completions.create(
        messages=messages,
        model=model_name,
        temperature=0.7,
        max_tokens=1024
    )

    content = response.choices[0].message.content

    # Extract reasoning if present
    if show_thinking:
        match = re.search(r"<think>(.*?)</think>(.*)", content, re.DOTALL)
        if match:
            return {"thinking": match.group(1).strip(), "answer": match.group(2).strip()}
    return content

# Example usage
query = "Plan a 5-day cultural trip to Kyoto in April"
result = plan_trip_with_reasoning(query, show_thinking=True)

print("🗺️ Query:", query)
if isinstance(result, dict):
    print("\n🧠 Thinking Process:", result["thinking"])
    print("\n📝 Final Answer:", result["answer"])
else:
    print("\n📝 Response:", result)

## 3. Technical Problem Solving 💻

Showcase coding/optimization capabilities:

In [ ]:
def solve_technical_problem(problem):
    """Solve complex technical problems with structured reasoning"""
    response = openai_client.chat.completions.create(
        messages=[
            {"role": "user", "content": rf"{problem} Please reason step by step, and put your final answer within \boxed{{}}."}
        ],
        model=model_name,
        temperature=0.3,
        max_tokens=2048
    )

    return response.choices[0].message.content

# Database optimization example
problem = """How can I optimize a PostgreSQL database handling 10k transactions/second?
Consider indexing strategies, hardware requirements, and query optimization."""

print("🔧 Problem:", problem)
print("\n⚙️ Solution:", solve_technical_problem(problem))

## 4. Best Practices & Considerations

1. **Reasoning Handling**: Use regex to separate <think> content from final answers
2. **Safety**: Built-in content filtering - handle errors for violations
3. **Performance**:
   - Max tokens: up to 1024 (travel) / 2048 (technical) in the examples above
   - Rate limit: 200K tokens/minute
4. **Cost**: Pay-as-you-go with serverless deployment
5. **Streaming**: Implement response streaming for long completions

```python
# Streaming example
response = openai_client.chat.completions.create(..., stream=True)
for chunk in response:
    print(chunk.choices[0].delta.content or "", end="")
```

## 🎯 Key Takeaways
- Leverage 128K context for detailed analysis
- Extract reasoning steps for debugging/analysis
- Combine with Azure AI Content Safety for production
- Monitor token usage via response.usage

> Always validate model outputs for critical applications!